# Avito: кандидатогенерация для поиска услуг (Recall@50)

Задача: по тексту запроса и его признакам выбрать из корпуса `benchmark_items.parquet` до 50 объявлений так, чтобы в них попало как можно больше подходящих. Порядок внутри 50 не важен, считается только полнота.

Что в итоге получилось (подробности ниже по ноутбуку):

- ищем по всему корпусу, каждому объявлению считаем общий скор как взвешенную сумму нескольких сигналов;
- сигналы: BM25 по тексту и по заголовку, вероятность локации объявления при локации поиска (посчитана по train), расстояние, совпадение фильтров из `search_infm_params_text` с параметрами объявления, подкатегория и расширение запроса по похожим запросам из train, эмбеддинги готовой модели и эмбеддинги той же модели, дообученной на парах "запрос - объявление" из train;
- веса подобраны по Recall@50 на своей валидации (часть train) и проверены на второй отложенной выборке;
- первая версия решения получила на платформе 0.877, хотя на валидации было 0.907. Разбор показал, что корпус benchmark собран вокруг самих запросов и моя валидация была слишком легкой. В разделе 6 валидация переделана так, чтобы быть похожей на benchmark (первая версия получает на ней 0.884), и уже на ней решение улучшено до 0.894 на отложенной выборке;
- в конце `answer.csv` проходит проверки из чек-листа.

Запуск: положить три parquet-файла в папку `dataset/` и выполнить ноутбук сверху вниз. На 8-ядерном CPU без GPU полный прогон с нуля занимает около 85 минут: эмбеддинги примерно 23 минуты, дообучение модели 17 минут, прогон валидации 30 минут. Повторный прогон с готовым кэшем идет около 45 минут. Эмбеддинги и дообученная модель сохраняются в `cache/`, поэтому повторный прогон быстрее.

## 0. Импорты и настройки

In [1]:
import os
import re
import time
import hashlib
import random

import numpy as np
import pandas as pd
import scipy.sparse as sp
import snowballstemmer
import torch
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import normalize
from sentence_transformers import SentenceTransformer

SEED = 42
DATA_DIR = 'dataset'
CACHE_DIR = 'cache'
EMB_MODEL_NAME = 'sergeyzh/rubert-tiny-turbo'

os.makedirs(CACHE_DIR, exist_ok=True)
np.random.seed(SEED)
torch.manual_seed(SEED)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.width', 200)
start_time = time.time()

## 1. Загрузка данных и проверка типов

Сначала проверяю, что файлы на месте и колонки называются так, как в описании. Отдельно смотрю на ID: они должны остаться строками длиной 16 символов. `item_id` состоит из `0-9a-f`, а у `query_id` в данных встречаются и заглавные латинские буквы, поэтому для него проверяю только длину и тип.

In [2]:
for name in ['train.parquet', 'benchmark_queries.parquet', 'benchmark_items.parquet']:
    path = os.path.join(DATA_DIR, name)
    if not os.path.exists(path):
        raise FileNotFoundError(f'Не найден файл {path}')

train = pd.read_parquet(os.path.join(DATA_DIR, 'train.parquet'))
queries = pd.read_parquet(os.path.join(DATA_DIR, 'benchmark_queries.parquet'))
items = pd.read_parquet(os.path.join(DATA_DIR, 'benchmark_items.parquet'))
print('train:', train.shape)
print('queries:', queries.shape)
print('items:', items.shape)

train: (497673, 19)
queries: (2452, 6)
items: (189212, 14)


In [3]:
search_cols = ['search_query', 'search_location_id', 'search_is_delivery_search', 'search_infm_params_text', 'search_category']
item_cols = ['item_id', 'item_title_raw', 'item_description_raw', 'item_infm_params_text', 'item_category_id',
             'item_microcat_id', 'item_price', 'item_rating', 'item_rating_reviews_count', 'item_location_id',
             'item_latitude', 'item_longitude', 'item_is_phone_hidden', 'item_is_message_forbidden']

missing = {
    'train': set(search_cols + item_cols) - set(train.columns),
    'queries': set(['query_id'] + search_cols) - set(queries.columns),
    'items': set(item_cols) - set(items.columns),
}
print(missing)
assert all(len(m) == 0 for m in missing.values())

{'train': set(), 'queries': set(), 'items': set()}


In [4]:
def check_ids(ids, name, hex_only):
    assert ids.map(type).eq(str).all(), name
    assert ids.str.len().eq(16).all(), name
    if hex_only:
        assert ids.str.fullmatch('[0-9a-f]{16}').all(), name
    print(f'{name}: dtype={ids.dtype}, пример={ids.iloc[0]}, пропусков={ids.isna().sum()}')


check_ids(queries['query_id'], 'query_id', hex_only=False)
check_ids(items['item_id'], 'item_id (корпус)', hex_only=True)
check_ids(train['item_id'], 'item_id (train)', hex_only=True)

query_id: dtype=str, пример=70DfDUpwjxB4lzFd, пропусков=0
item_id (корпус): dtype=str, пример=111eb8b979577d79, пропусков=0
item_id (train): dtype=str, пример=e8b685dffe1a408e, пропусков=0


Текстовые поля иногда пустые (например, 35 описаний в корпусе). Чтобы дальше ничего не падало на `None`, сразу заменяю пропуски в текстах на пустую строку.

In [5]:
for df in [train, queries]:
    for col in ['search_query', 'search_infm_params_text']:
        df[col] = df[col].fillna('')
for df in [train, items]:
    for col in ['item_title_raw', 'item_description_raw', 'item_infm_params_text']:
        df[col] = df[col].fillna('')

print(queries.isna().sum().sum(), items[['item_title_raw', 'item_description_raw', 'item_infm_params_text']].isna().sum().sum())

0 0


## 2. EDA

In [6]:
print('строк в train:', len(train))
print('уникальных текстов запросов в train:', train['search_query'].nunique())
print('уникальных комбинаций всех search_* полей в train:', train[search_cols].drop_duplicates().shape[0])
print('уникальных объявлений в train:', train['item_id'].nunique())
print()
print('запросов в benchmark:', len(queries), '| уникальных query_id:', queries['query_id'].nunique(), '| уникальных текстов:', queries['search_query'].nunique())
print('объявлений в корпусе:', len(items), '| уникальных item_id:', items['item_id'].nunique())
print('объявлений корпуса, которые встречаются в train:', items['item_id'].isin(set(train['item_id'])).sum())
seen_share = queries['search_query'].isin(set(train['search_query'])).mean()
print('доля запросов benchmark, текст которых был в train:', round(seen_share, 3))

строк в train: 497673
уникальных текстов запросов в train: 74529


уникальных комбинаций всех search_* полей в train: 354463
уникальных объявлений в train: 344825

запросов в benchmark: 2452 | уникальных query_id: 2452 | уникальных текстов: 2452
объявлений в корпусе: 189212 | уникальных item_id: 189212


объявлений корпуса, которые встречаются в train: 18142


доля запросов benchmark, текст которых был в train: 0.37


In [7]:
lengths = pd.DataFrame({
    'слов в запросе': queries['search_query'].str.split().str.len().describe(),
    'символов в заголовке': items['item_title_raw'].str.len().describe(),
    'символов в описании': items['item_description_raw'].str.len().describe(),
    'символов в параметрах': items['item_infm_params_text'].str.len().describe(),
})
lengths.round(1)

,слов в запросе,символов в заголовке,символов в описании,символов в параметрах
count,2452.0,189212.0,189212.0,189212.0
mean,3.2,33.6,1404.9,973.9
std,1.3,12.2,1284.3,875.9
min,1.0,1.0,0.0,10.0
25%,2.0,24.0,422.0,453.0
50%,3.0,35.0,1054.0,738.0
75%,4.0,44.0,1978.0,1206.0
max,10.0,100.0,8494.0,19068.0


In [8]:
print('пустой search_infm_params_text в train:', round((train['search_infm_params_text'] == '').mean(), 3))
print('пустой search_infm_params_text в benchmark:', round((queries['search_infm_params_text'] == '').mean(), 3))
print('пустых описаний в корпусе:', (items['item_description_raw'] == '').sum())
print('пустых заголовков в корпусе:', (items['item_title_raw'] == '').sum())
print('запросов из одного слова в benchmark:', (queries['search_query'].str.split().str.len() == 1).sum())

пустой search_infm_params_text в train: 0.33
пустой search_infm_params_text в benchmark: 0.631
пустых описаний в корпусе: 35
пустых заголовков в корпусе: 0
запросов из одного слова в benchmark: 109


In [9]:
print(pd.crosstab(train['search_category'], train['item_category_id']))
print()
print('search_category в benchmark:', queries['search_category'].value_counts().to_dict())
print('item_category_id в корпусе (топ-10):', items['item_category_id'].value_counts().head(10).to_dict())

item_category_id  10   19   27   28   40   81   112     114
search_category                                            
0                   0    0    0    0    0    0    0      34
10                  0    0    0    0    0    0    0       2
81                  0    0    0    0    0    0    0       2
114                 1    2    4    2    2    2    2  497620

search_category в benchmark: {114: 2230, 0: 222}
item_category_id в корпусе (топ-10): {114: 187336, 19: 357, 112: 141, 40: 122, 33: 103, 10: 98, 20: 84, 25: 82, 36: 74, 111: 74}


In [10]:
print('дублей строк (запрос + объявление) в train:', train.duplicated(search_cols + ['item_id']).sum())
print('объявлений в корпусе с таким же заголовком и описанием, как у другого:', items.duplicated(['item_title_raw', 'item_description_raw']).sum())

дублей строк (запрос + объявление) в train: 30619


объявлений в корпусе с таким же заголовком и описанием, как у другого: 18754


In [11]:
same_loc = (train['search_location_id'] == train['item_location_id']).mean()
print('доля пар в train, где локация поиска совпадает с локацией объявления:', round(same_loc, 3))
print('локации поиска, у которых чаще всего выбирают объявление из другой локации:')
print(train.loc[train['search_location_id'] != train['item_location_id'], 'search_location_id'].value_counts().head(5))

доля пар в train, где локация поиска совпадает с локацией объявления: 0.831
локации поиска, у которых чаще всего выбирают объявление из другой локации:
search_location_id
107620    12277
621540     6839
107621     6310
637640     2873
637680     1528
Name: count, dtype: int64


In [12]:
train['search_infm_params_text'].value_counts().head(12)

search_infm_params_text
                                                                                                        163999
Вид услуги Автосервис, аренда Тип услуги Автосервис                                                      19654
Вид услуги Ремонт и отделка                                                                              15586
Вид услуги Красота, здоровье                                                                             15355
Вид услуги Праздники, мероприятия                                                                        12474
Тип услуги автосервиса Диагностика и ремонт авто Вид услуги Автосервис, аренда Тип услуги Автосервис     11644
Тип услуги Маникюр, педикюр Вид услуги Красота, здоровье                                                 11589
Вид услуги Обучение, курсы                                                                               11080
Вид услуги                                                                              

Что видно из EDA:

- Почти все запросы и объявления в train из категории 114 (услуги). В корпусе есть около 1.9 тыс. объявлений из других категорий, это скорее шум. У 222 запросов benchmark `search_category = 0`, но в train у таких запросов объявления все равно из 114.
- Запросы короткие, медиана 3 слова. Описания объявлений длинные, в параметрах много служебного текста (график работы и т.п.).
- Все тексты запросов в benchmark разные, и только около 37% из них встречались в train. Значит, простое запоминание пар "запрос - объявление" не спасет, нужен поиск по тексту.
- Объявления корпуса почти не пересекаются с train (около 18 тыс. из 189 тыс.).
- Очень сильный сигнал - локация: в 83% пар локация поиска совпадает с локацией объявления. Остальное в основном "регион -> город" (например, поиск по области, а объявление в областном центре). Такие переходы можно посчитать по train.
- `search_infm_params_text` - это фильтры поиска ("Вид услуги ...", "Тип услуги ..."). У 37% запросов benchmark они заполнены, и их можно сверять с `item_infm_params_text`.
- Есть дубли строк в train и объявления с одинаковым текстом в корпусе. Дубли строк при группировке по запросу схлопываются, а дубли объявлений проверю в конце на ответе.

## 3. Подготовка текста

Токенизация простая: нижний регистр, `ё -> е`, слова из букв и цифр, затем стемминг Snowball для русского (чтобы "дровах" и "дрова" совпали). Эмодзи и прочие символы просто выкидываются, пустая строка дает пустой результат.

In [13]:
stemmer = snowballstemmer.stemmer('russian')
token_re = re.compile(r'[a-zа-я0-9]+')
stem_cache = {}


def stem_text(text):
    words = token_re.findall(text.lower().replace('ё', 'е'))
    result = []
    for w in words:
        if w not in stem_cache:
            stem_cache[w] = stemmer.stemWord(w)
        result.append(stem_cache[w])
    return ' '.join(result)


examples = ['Баня на дровах', 'ремонт iPhone 11 \U0001F600', '', '   ', '№1!!! Ёлка-палка, 5-10 м²']
for example in examples:
    print(repr(stem_text(example)))

'бан на дров'
'ремонт iphone 11'
''
''
'1 елк палк 5 10 м'


## 4. Валидация: сплит и метрика

У benchmark нет разметки, поэтому качество проверяю на части train.

- Делю по уникальным комбинациям всех `search_*` полей, а не по строкам: одна и та же комбинация не может оказаться и в обучающей, и в валидационной части. 40% комбинаций уходит в валидацию. Доля большая, потому что из валидационной части потом берутся "похожие" объявления для усложнения валидации (раздел 6), а для статистики по train хватает и оставшихся 60%.
- Все объявления, выбранные по одной комбинации, собираю в один валидационный запрос с несколькими релевантными `item_id`.
- В benchmark у каждого запроса свой текст, и только 37% текстов были в train. Чтобы валидация была похожа, беру по одной комбинации на текст и набираю запросы так, чтобы доля "виденных" текстов (есть в обучающей части) тоже была около 37%.
- Делаю две непересекающиеся выборки по ~3000 запросов: A для подбора весов и B для честной проверки.
- Корпус для поиска на валидации: весь `benchmark_items`, релевантные объявления валидационных запросов и "похожие" объявления, которые подбираются в разделе 6. То есть нужные объявления ищутся среди ~270 тыс. других, как и в настоящей задаче.
- Вся статистика (переходы локаций, похожие запросы и т.д.) на валидации считается только по обучающей части.

In [14]:
def recall_at_50(predictions, relevant):
    scores = []
    for pred, rel in zip(predictions, relevant):
        rel = set(rel)
        scores.append(len(set(pred[:50]) & rel) / len(rel))
    return float(np.mean(scores))


example_pred = [['a1'], ['b1', 'x1'], ['z1']]
example_rel = [['a1'], ['b1', 'b2'], ['c1']]
print('пример из задания:', recall_at_50(example_pred, example_rel))
assert abs(recall_at_50(example_pred, example_rel) - 0.5) < 1e-12
assert recall_at_50([[str(i) for i in range(100)]], [['0', '75']]) == 0.5
assert recall_at_50([['a', 'a', 'b']], [['a', 'b']]) == 1.0

пример из задания: 0.5


In [15]:
combos = train[search_cols].drop_duplicates().reset_index(drop=True)
rng = np.random.RandomState(SEED)
combos['is_val'] = rng.rand(len(combos)) < 0.4
train = train.merge(combos, on=search_cols, how='left')
assert train['is_val'].notna().all() and train['item_id'].dtype == items['item_id'].dtype

train_part = train[~train['is_val']].reset_index(drop=True)
val_part = train[train['is_val']].reset_index(drop=True)
print('обучающая часть:', len(train_part), 'строк | валидационная часть:', len(val_part), 'строк')

val_queries = val_part.groupby(search_cols)['item_id'].apply(lambda s: sorted(set(s))).reset_index(name='relevant')
val_queries = val_queries.sample(frac=1, random_state=SEED).drop_duplicates('search_query').reset_index(drop=True)
val_queries['seen'] = val_queries['search_query'].isin(set(train_part['search_query']))

n_unseen = 1900
n_seen = int(round(n_unseen * seen_share / (1 - seen_share)))
unseen_q = val_queries[~val_queries['seen']]
seen_q = val_queries[val_queries['seen']]
val_a = pd.concat([unseen_q.iloc[:n_unseen], seen_q.iloc[:n_seen]]).reset_index(drop=True)
val_b = pd.concat([unseen_q.iloc[n_unseen:2 * n_unseen], seen_q.iloc[n_seen:2 * n_seen]]).reset_index(drop=True)
print('A:', len(val_a), 'запросов, доля виденных', round(val_a['seen'].mean(), 3), ', релевантных на запрос', round(val_a['relevant'].str.len().mean(), 2))
print('B:', len(val_b), 'запросов, доля виденных', round(val_b['seen'].mean(), 3), ', релевантных на запрос', round(val_b['relevant'].str.len().mean(), 2))
assert not set(val_a['search_query']) & set(val_b['search_query'])

обучающая часть: 299757 строк | валидационная часть: 197916 строк


A: 3015 запросов, доля виденных 0.37 , релевантных на запрос 1.12
B: 3015 запросов, доля виденных 0.37 , релевантных на запрос 1.09


Собираю все объявления, которые понадобятся дальше: `benchmark_items`, релевантные объявления выборок A и B и объявления из валидационной части, которых нет в обучающей части. Из последних в разделе 6 выбираются "похожие" объявления для валидации.

In [16]:
bench_ids = set(items['item_id'])
val_relevant_ids = set(i for lst in pd.concat([val_a, val_b])['relevant'] for i in lst)
val_items = val_part.drop_duplicates('item_id')[item_cols]
val_items = val_items[~val_items['item_id'].isin(bench_ids)]
cold_ids = set(val_items['item_id']) - set(train_part['item_id'])
extra_items = val_items[val_items['item_id'].isin(val_relevant_ids | cold_ids)]
all_corpus = pd.concat([items[item_cols], extra_items]).reset_index(drop=True)
n_bench_items = len(items)
assert all_corpus['item_id'].is_unique
assert (all_corpus['item_id'].iloc[:n_bench_items].values == items['item_id'].values).all()
print('всего объявлений:', len(all_corpus), '| из benchmark:', n_bench_items, '| релевантных не из benchmark:', len(val_relevant_ids - bench_ids), '| нет в обучающей части:', len(cold_ids))

всего объявлений: 301779 | из benchmark: 189212 | релевантных не из benchmark: 6170 | нет в обучающей части: 110834


Стемминг текстов корпуса. Первые 189 212 строк `all_corpus` - это ровно `benchmark_items`, поэтому для финального ответа потом просто беру эту часть.

In [17]:
t = time.time()
all_corpus['title_st'] = [stem_text(x) for x in all_corpus['item_title_raw']]
all_corpus['full_st'] = [a + ' ' + stem_text(b) + ' ' + stem_text(c) for a, b, c in
                         zip(all_corpus['title_st'], all_corpus['item_description_raw'], all_corpus['item_infm_params_text'])]
title_st_map = {}
for item_id, title in zip(train['item_id'], train['item_title_raw']):
    if item_id not in title_st_map:
        title_st_map[item_id] = stem_text(title)
train['title_st'] = train['item_id'].map(title_st_map)
train_part['title_st'] = train_part['item_id'].map(title_st_map)
for df in [val_a, val_b, queries]:
    df['query_st'] = [stem_text(x) for x in df['search_query']]
val_ab = pd.concat([val_a.assign(part='A'), val_b.assign(part='B')]).reset_index(drop=True)
print('стемминг занял', round(time.time() - t), 'сек')

стемминг занял 91 сек


## 5. Эмбеддинги готовой модели

Беру открытую модель `sergeyzh/rubert-tiny-turbo` (маленький русский BERT, 312-мерные эмбеддинги). Она быстрая на CPU, а большие модели вроде multilingual-e5 на этом железе считали бы корпус больше часа. Модель скачивается с Hugging Face один раз при первом запуске, дальше все считается локально без обращений к сети.

Текст объявления для эмбеддинга: заголовок + первые 300 символов описания. Результаты сохраняю в `cache/`, в имя файла добавляю хэш от входных текстов, чтобы случайно не взять устаревший кэш.

In [18]:
base_model = SentenceTransformer(EMB_MODEL_NAME, device='cpu')


def encode(model, texts, name, max_len):
    key = hashlib.md5(('\n'.join(texts) + name).encode('utf-8')).hexdigest()[:12]
    path = os.path.join(CACHE_DIR, f'{name}_{key}.npy')
    if os.path.exists(path):
        return np.load(path)
    model.max_seq_length = max_len
    emb = model.encode(texts, batch_size=128, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)
    emb = emb.astype(np.float32)
    np.save(path, emb)
    return emb


t = time.time()
item_texts = (all_corpus['item_title_raw'] + '. ' + all_corpus['item_description_raw'].str.slice(0, 300)).tolist()
item_emb = encode(base_model, item_texts, 'items', 128)
all_query_texts = sorted(set(train['search_query']) | set(val_ab['search_query']) | set(queries['search_query']))
query_emb = encode(base_model, all_query_texts, 'queries', 128)
query_emb_pos = pd.Series(np.arange(len(all_query_texts)), index=all_query_texts)
print(item_emb.shape, query_emb.shape, 'за', round(time.time() - t), 'сек')

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

(301779, 312) (76074, 312) за 1041 сек


## 6. Почему первая валидация была слишком легкой и как это исправлено

Первая версия этого ноутбука строила корпус для валидации просто как `benchmark_items` + релевантные объявления валидационных запросов. На такой валидации получалось Recall@50 = 0.907, а на платформе та же модель получила **0.877**. Разница большая, значит валидация была устроена не так, как benchmark.

Чтобы понять разницу, считаю для каждого запроса, сколько в корпусе объявлений из **той же локации**, которые **похожи на запрос по смыслу** (косинус эмбеддингов больше 0.7 и больше 0.8). Для запросов benchmark считаю по `benchmark_items`, для валидационных запросов - по тому корпусу, который был в первой версии.

In [19]:
item_loc = all_corpus['item_location_id'].values


def close_items(qdf, positions):
    q = query_emb[query_emb_pos.loc[qdf['search_query']].values]
    e = item_emb[positions]
    locs = item_loc[positions]
    n7, n8 = [], []
    for start in range(0, len(qdf), 500):
        sims = q[start:start + 500] @ e.T
        same = locs[None, :] == qdf['search_location_id'].values[start:start + 500, None]
        n7.append(((sims > 0.7) & same).sum(axis=1))
        n8.append(((sims > 0.8) & same).sum(axis=1))
    return np.concatenate(n7), np.concatenate(n8)


bench_pos = np.arange(n_bench_items)
old_val_pos = np.flatnonzero((np.arange(len(all_corpus)) < n_bench_items) | all_corpus['item_id'].isin(val_relevant_ids).values)
rows = []
for name, qdf, pos in [('benchmark', queries, bench_pos), ('валидация, первая версия', val_ab, old_val_pos)]:
    n7, n8 = close_items(qdf, pos)
    rows.append({'запросы': name, 'медиана, косинус > 0.7': np.median(n7), 'медиана, косинус > 0.8': np.median(n8), 'среднее, косинус > 0.8': n8.mean().round(1)})
pd.DataFrame(rows)

,запросы,"медиана, косинус > 0.7","медиана, косинус > 0.8","среднее, косинус > 0.8"
0,benchmark,333.5,7.0,19.2
1,"валидация, первая версия",190.0,1.0,9.7


У запросов benchmark рядом (в той же локации и по смыслу) заметно больше конкурентов, чем у валидационных. При этом если считать похожие объявления по всему корпусу без учета локации, разницы почти нет. То есть корпус benchmark, похоже, собран вокруг самих запросов: в нем много объявлений, которые подходили бы под запрос и находятся в том же городе (например, все, что показывалось в выдаче по этому запросу). Нужное объявление приходится отличать от них, а в первой валидации таких конкурентов почти не было, поэтому локация решала почти все.

Исправление: для каждого валидационного запроса добавляю в корпус "похожие" объявления:

- берутся только объявления из валидационной части, которых нет в обучающей части (как и в benchmark, где почти все объявления новые);
- только из локаций, куда при таком поиске реально уходят пользователи: та же локация или локации, куда по статистике обучающей части уходит хотя бы 5% выбора;
- 30 лучших по BM25 и 30 лучших по эмбеддингам, кроме релевантных объявлений самого запроса.

Число 30 подобрано так, чтобы плотность похожих объявлений стала близка к benchmark. Проверка калибровки - строка "первая версия" в таблице раздела 9: веса первого отправленного решения на новой валидации получают около 0.88, на платформе было 0.877, а на старой валидации было 0.907. Значит, новая валидация намного ближе к реальности.

In [20]:
def bm25_matrix(texts, k1=1.2, b=0.75):
    vectorizer = CountVectorizer(analyzer=str.split, dtype=np.float32)
    tf = vectorizer.fit_transform(texts).tocsr()
    n_docs = tf.shape[0]
    df = np.bincount(tf.indices, minlength=tf.shape[1])
    idf = np.log(1 + (n_docs - df + 0.5) / (df + 0.5))
    doc_len = np.asarray(tf.sum(axis=1)).ravel()
    avg_len = max(doc_len.mean(), 1.0)
    rows = np.repeat(np.arange(n_docs), np.diff(tf.indptr))
    x = tf.data
    tf.data = (x * (k1 + 1) / (x + k1 * (1 - b + b * doc_len[rows] / avg_len)) * idf[tf.indices]).astype(np.float32)
    return tf.T.tocsr(), vectorizer.vocabulary_


def query_matrix(texts, vocabulary):
    return CountVectorizer(analyzer=str.split, vocabulary=vocabulary, binary=True, dtype=np.float32).transform(texts)

In [21]:
K_NEG = 30
t = time.time()
counts = train_part.groupby(['search_location_id', 'item_location_id']).size().reset_index(name='n')
counts['p'] = counts['n'] / counts.groupby('search_location_id')['n'].transform('sum')
allowed_locs = counts[counts['p'] >= 0.05].groupby('search_location_id')['item_location_id'].apply(set).to_dict()

cold_pos = np.flatnonzero(all_corpus['item_id'].isin(cold_ids).values)
cold_item_ids = all_corpus['item_id'].values[cold_pos]
cold_index = pd.Series(np.arange(len(cold_pos)), index=cold_item_ids)
cold_loc = item_loc[cold_pos]
cold_emb = item_emb[cold_pos]
cold_bm25, cold_vocab = bm25_matrix(all_corpus['full_st'].values[cold_pos])

negatives = set()
for start in range(0, len(val_ab), 200):
    batch = val_ab.iloc[start:start + 200]
    bm = (query_matrix(batch['query_st'], cold_vocab) @ cold_bm25).toarray()
    em = query_emb[query_emb_pos.loc[batch['search_query']].values] @ cold_emb.T
    for row in range(len(batch)):
        q = batch.iloc[row]
        locs = allowed_locs.get(q['search_location_id'], set()) | {q['search_location_id']}
        ok = np.isin(cold_loc, list(locs))
        ok[cold_index.reindex(q['relevant']).dropna().astype(int).values] = False
        for scores in [np.where(ok & (bm[row] > 0), bm[row], -np.inf), np.where(ok, em[row], -np.inf)]:
            top = np.argsort(-scores, kind='stable')[:K_NEG]
            negatives.update(cold_pos[top[np.isfinite(scores[top])]].tolist())

keep = np.zeros(len(all_corpus), dtype=bool)
keep[:n_bench_items] = True
keep[all_corpus['item_id'].isin(val_relevant_ids).values] = True
keep[sorted(negatives)] = True
val_corpus = all_corpus[keep].reset_index(drop=True)
print('добавлено похожих объявлений:', len(negatives), '| корпус для валидации:', len(val_corpus), '| за', round(time.time() - t), 'сек')

n7, n8 = close_items(val_ab, np.flatnonzero(keep))
print('новая валидация: медиана косинус > 0.7:', np.median(n7), '| косинус > 0.8:', np.median(n8), '| среднее косинус > 0.8:', n8.mean().round(1))

добавлено похожих объявлений: 79786 | корпус для валидации: 271753 | за 42 сек


новая валидация: медиана косинус > 0.7: 304.5 | косинус > 0.8: 2.0 | среднее косинус > 0.8: 14.5


## 7. Дообучение модели эмбеддингов

На новой валидации стало видно, что важно не только найти объявления в нужном городе, но и точнее понять, какое из похожих объявлений подходит под запрос. Готовая модель эмбеддингов обучалась на общих текстах, а в train есть почти полмиллиона примеров "запрос - выбранное объявление". Дообучаю ту же `rubert-tiny-turbo` на этих парах.

Как устроено обучение:

- пары берутся только из обучающей части (60% train), иначе на валидации была бы утечка; эта же модель используется и для финального ответа;
- для каждого текста запроса беру до 3 разных выбранных объявлений, всего около 100 тыс. пар; текст объявления - заголовок + первые 100 символов описания;
- в батче 64 пары. Для каждого запроса "правильное" объявление - его пара, а остальные 63 объявления батча считаются неправильными. Модель учится, чтобы косинус запроса со своей парой был выше, чем с чужими (это обычная схема для таких моделей, кросс-энтропия по матрице косинусов);
- в одном батче нет одинаковых текстов запросов, чтобы не наказывать модель за правильный ответ;
- одна эпоха, AdamW, lr = 5e-5 с разогревом и линейным спадом, все случайности зафиксированы, включен детерминированный режим PyTorch.

На CPU это занимает около 25 минут. Готовая модель сохраняется в `cache/`, при повторном запуске обучение пропускается.

In [22]:
pairs = train_part.drop_duplicates(['search_query', 'item_id']).sample(frac=1, random_state=SEED)
pairs['r'] = pairs.groupby('search_query').cumcount()
pairs = pairs[pairs['r'] < 3].copy()
pairs['text'] = pairs['item_title_raw'] + '. ' + pairs['item_description_raw'].str.slice(0, 100)
batches = []
for r, group in pairs.groupby('r'):
    group = group.sample(frac=1, random_state=r)
    for start in range(0, len(group) - 63, 64):
        batches.append(group.iloc[start:start + 64])
random.Random(SEED).shuffle(batches)
print('пар для обучения:', len(pairs), '| батчей:', len(batches))
assert all(b['search_query'].is_unique for b in batches)

пар для обучения: 86834 | батчей: 1355


In [23]:
def encode_for_training(model, texts, max_len):
    model.max_seq_length = max_len
    features = model.preprocess(texts)
    return torch.nn.functional.normalize(model(features)['sentence_embedding'], dim=-1)


def fine_tune(batches, path):
    torch.manual_seed(SEED)
    torch.use_deterministic_algorithms(True)
    model = SentenceTransformer(EMB_MODEL_NAME, device='cpu')
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
    n_steps = len(batches)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda step: min(1.0, (step + 1) / 100) * max(0.0, 1 - step / n_steps))
    model.train()
    for step, batch in enumerate(batches):
        q = encode_for_training(model, batch['search_query'].tolist(), 32)
        d = encode_for_training(model, batch['text'].tolist(), 48)
        sim = q @ d.T * 20
        labels = torch.arange(len(batch))
        loss = (torch.nn.functional.cross_entropy(sim, labels) + torch.nn.functional.cross_entropy(sim.T, labels)) / 2
        loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        if step % 300 == 0:
            print('шаг', step, 'из', n_steps, '| loss', round(loss.item(), 3))
    model.eval()
    model.save(path)
    torch.use_deterministic_algorithms(False)


ft_key = hashlib.md5(pd.util.hash_pandas_object(pairs[['search_query', 'text']], index=False).values.tobytes()).hexdigest()[:12]
ft_path = os.path.join(CACHE_DIR, f'ft_model_{ft_key}')
t = time.time()
if not os.path.exists(ft_path):
    fine_tune(batches, ft_path)
ft_model = SentenceTransformer(ft_path, device='cpu')
print('модель готова за', round(time.time() - t), 'сек')

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

шаг 0 из 1355 | loss 2.647


шаг 300 из 1355 | loss 1.344


шаг 600 из 1355 | loss 0.67


шаг 900 из 1355 | loss 0.76


шаг 1200 из 1355 | loss 0.752


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

модель готова за 989 сек


In [24]:
t = time.time()
ft_texts = (val_corpus['item_title_raw'] + '. ' + val_corpus['item_description_raw'].str.slice(0, 100)).tolist()
ft_item_emb = encode(ft_model, ft_texts, 'ft_items', 48)
ft_query_emb = encode(ft_model, all_query_texts, 'ft_queries', 32)
print(ft_item_emb.shape, ft_query_emb.shape, 'за', round(time.time() - t), 'сек')

(271753, 312) (76074, 312) за 358 сек


## 8. Сигналы для скоринга

Для каждого запроса считаю скор для всех объявлений корпуса сразу (матрицами, батчами по 200 запросов) и беру 50 лучших. Все, что считается по train, собирается функцией `build_index`: на валидации ей передается только обучающая часть, а для финального ответа - весь train.

Сигналы:

1. `bm_full` - BM25 по заголовку + описанию + параметрам (стеммированные слова).
2. `bm_title` - BM25 только по заголовку.
3. `loc` - вероятность локации объявления при локации поиска, посчитанная по train: P(item_location | search_location). В скор идет её логарифм. Для локации поиска, которой нет в train, строка нулевая, и этот сигнал просто одинаковый для всех объявлений.
4. `dist` - расстояние (км) от "центра" локации поиска (медиана координат выбранных объявлений в train) до координат объявления.
5. `nl` - "нелокальность" запроса: по 20 похожим запросам из train смотрю, какая доля выборов была в другом месте (маловероятная локация или дальше 100 км). Для запросов вроде "база отдыха", "геологические изыскания", "аренда авто в стамбуле" это заметная доля. У таких запросов штраф за локацию ослабляется.
6. `flt` - фильтры: из `search_infm_params_text` достаю "Вид услуги ..." и "Тип услуги ..." и проверяю, есть ли они в `item_infm_params_text`. +1 если все совпали, -1 если нет, 0 если фильтров нет.
7. `mc` - подкатегория: по тем же 20 похожим запросам из train смотрю, из каких `item_microcat_id` там выбирали объявления.
8. `exp` - расширение запроса: заголовки объявлений, выбранных по похожим запросам, сравниваются с заголовками корпуса (TF-IDF косинус). Так находятся синонимы.
9. `emb` - косинус эмбеддингов готовой модели.
10. `emb_ft` - косинус эмбеддингов дообученной модели.
11. `cat` - объявление из категории 114 (услуги) или нет.

Итоговый скор - взвешенная сумма этих сигналов, веса подбираются ниже по Recall@50.

In [25]:
def to_float(values):
    return pd.Series(values).astype(float).values


def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    h = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(np.clip(h, 0, 1)))

In [26]:
all_params = pd.concat([train['item_infm_params_text'], items['item_infm_params_text']])
stop_words = r'(?: Место оказания| Тип услуги| Вид услуги| Тип стоимости|$)'
filter_values = {}
for key in ['Вид услуги', 'Тип услуги']:
    counts = all_params.str.extract(key + r' (.*?)' + stop_words)[0].value_counts()
    filter_values[key] = sorted([v for v in counts[counts >= 5].index if v and 'Место' not in v], key=len, reverse=True)


def parse_filters(params_text):
    found = []
    for key, values in filter_values.items():
        for v in values:
            if key + ' ' + v in params_text:
                found.append(key + ' ' + v)
                break
    return found


print({k: len(v) for k, v in filter_values.items()})
for text in ['', 'Вид услуги', 'Онлайн-запись Тип услуги Маникюр, педикюр Вид услуги Красота, здоровье', 'Рейтинг пользователя 4 звезды и выше']:
    print(repr(text), '->', parse_filters(text))

{'Вид услуги': 26, 'Тип услуги': 2461}
'' -> []
'Вид услуги' -> []
'Онлайн-запись Тип услуги Маникюр, педикюр Вид услуги Красота, здоровье' -> ['Вид услуги Красота, здоровье', 'Тип услуги Маникюр, педикюр']
'Рейтинг пользователя 4 звезды и выше' -> []


In [27]:
sample = train.sample(50000, random_state=SEED)
sample_filters = sample['search_infm_params_text'].map({t: parse_filters(t) for t in sample['search_infm_params_text'].unique()})
has_filters = sample_filters.str.len() > 0
match = [all(f in p for f in fs) for fs, p in zip(sample_filters[has_filters], sample.loc[has_filters, 'item_infm_params_text'])]
print('строк с фильтрами:', round(has_filters.mean(), 3), '| фильтры совпали с параметрами объявления:', round(np.mean(match), 3))

строк с фильтрами: 0.644 | фильтры совпали с параметрами объявления: 0.979


In [28]:
def build_index(train_df, corpus, corpus_emb, corpus_ft_emb, k_neighbors=20):
    index = {'corpus': corpus, 'item_ids': corpus['item_id'].to_numpy(dtype=object), 'emb': corpus_emb, 'ft_emb': corpus_ft_emb, 'k': k_neighbors}
    index['bm_full'], index['vocab_full'] = bm25_matrix(corpus['full_st'].values)
    index['bm_title'], index['vocab_title'] = bm25_matrix(corpus['title_st'].values, b=0.5)

    search_locs = pd.Index(sorted(train_df['search_location_id'].unique()))
    item_locs = pd.Index(sorted(set(train_df['item_location_id']) | set(corpus['item_location_id'])))
    counts = train_df.groupby(['search_location_id', 'item_location_id']).size().reset_index(name='n')
    loc_prior = np.zeros((len(search_locs) + 1, len(item_locs)), dtype=np.float32)
    loc_prior[search_locs.get_indexer(counts['search_location_id']), item_locs.get_indexer(counts['item_location_id'])] = counts['n'].values
    loc_prior = loc_prior / np.maximum(loc_prior.sum(axis=1, keepdims=True), 1)
    index['search_locs'] = search_locs
    index['loc_prior'] = loc_prior
    index['item_loc_pos'] = item_locs.get_indexer(corpus['item_location_id'])

    lat = to_float(corpus['item_latitude'])
    lon = to_float(corpus['item_longitude'])
    loc_center = pd.DataFrame({'loc': corpus['item_location_id'].values, 'lat': lat, 'lon': lon}).groupby('loc')[['lat', 'lon']].median()
    index['item_lat'] = np.where(np.isnan(lat), loc_center['lat'].reindex(corpus['item_location_id']).values, lat)
    index['item_lon'] = np.where(np.isnan(lon), loc_center['lon'].reindex(corpus['item_location_id']).values, lon)
    click_lat = to_float(train_df['item_latitude'])
    click_lon = to_float(train_df['item_longitude'])
    clicks = pd.DataFrame({'loc': train_df['search_location_id'].values, 'lat': click_lat, 'lon': click_lon})
    index['search_center'] = clicks.groupby('loc')[['lat', 'lon']].median()

    pair_p = loc_prior[search_locs.get_indexer(train_df['search_location_id']), item_locs.get_indexer(train_df['item_location_id'])]
    center = index['search_center'].reindex(train_df['search_location_id'])
    click_km = np.nan_to_num(haversine(center['lat'].values, center['lon'].values, click_lat, click_lon), nan=0.0)
    train_df = train_df.assign(is_far=((pair_p < 0.01) | (click_km > 100)).astype(np.float32))

    past = train_df.groupby('search_query').agg(titles=('title_st', ' '.join), microcats=('item_microcat_id', list), far=('is_far', 'mean')).reset_index()
    index['past_emb'] = query_emb[query_emb_pos.loc[past['search_query']].values]
    index['past_far'] = past['far'].values.astype(np.float32)
    title_tfidf = TfidfVectorizer(analyzer=str.split, sublinear_tf=True, min_df=2, dtype=np.float32)
    title_tfidf.fit(corpus['title_st'])
    index['title_vecs'] = normalize(title_tfidf.transform(corpus['title_st'])).T.tocsr()
    index['past_titles'] = normalize(title_tfidf.transform(past['titles']))

    microcats = pd.Index(sorted(train_df['item_microcat_id'].unique()))
    rows = np.repeat(np.arange(len(past)), past['microcats'].str.len())
    cols = microcats.get_indexer(np.concatenate(past['microcats'].values))
    mc = sp.csr_matrix((np.ones(len(rows), dtype=np.float32), (rows, cols)), shape=(len(past), len(microcats)))
    index['past_microcats'] = normalize(mc, norm='l1')
    item_mc_pos = microcats.get_indexer(corpus['item_microcat_id'])
    item_mc_pos[item_mc_pos < 0] = len(microcats)
    index['item_mc_pos'] = item_mc_pos

    index['is_services'] = (corpus['item_category_id'].values == 114).astype(np.float32)
    index['filter_cache'] = {}
    return index

In [29]:
def filter_scores(index, params_texts):
    result = np.zeros((len(params_texts), len(index['item_ids'])), dtype=np.float32)
    for row, text in enumerate(params_texts):
        filters = parse_filters(text)
        if not filters:
            continue
        ok = np.ones(len(index['item_ids']), dtype=bool)
        for f in filters:
            if f not in index['filter_cache']:
                index['filter_cache'][f] = index['corpus']['item_infm_params_text'].str.contains(f, regex=False).values
            ok &= index['filter_cache'][f]
        result[row] = np.where(ok, 1, -1)
    return result


def compute_signals(index, batch):
    signals = {}
    signals['bm_full'] = (query_matrix(batch['query_st'], index['vocab_full']) @ index['bm_full']).toarray()
    signals['bm_title'] = (query_matrix(batch['query_st'], index['vocab_title']) @ index['bm_title']).toarray()

    q_emb = query_emb[query_emb_pos.loc[batch['search_query']].values]
    signals['emb'] = q_emb @ index['emb'].T
    signals['emb_ft'] = ft_query_emb[query_emb_pos.loc[batch['search_query']].values] @ index['ft_emb'].T

    k = index['k']
    sims = q_emb @ index['past_emb'].T
    nearest = np.argpartition(-sims, k, axis=1)[:, :k]
    weights = np.clip(np.take_along_axis(sims, nearest, axis=1), 0, None) ** 2
    rows = np.repeat(np.arange(len(batch)), k)
    neighbors = sp.csr_matrix((weights.ravel(), (rows, nearest.ravel())), shape=sims.shape)
    signals['exp'] = (normalize(neighbors @ index['past_titles']) @ index['title_vecs']).toarray()
    mc = normalize(neighbors @ index['past_microcats'], norm='l1').toarray()
    mc = np.hstack([mc, np.zeros((len(batch), 1), dtype=np.float32)])
    signals['mc'] = mc[:, index['item_mc_pos']]
    nl = (weights * index['past_far'][nearest]).sum(axis=1) / np.maximum(weights.sum(axis=1), 1e-9)
    signals['nl'] = np.broadcast_to(nl[:, None].astype(np.float32), signals['emb'].shape)

    loc_pos = index['search_locs'].get_indexer(batch['search_location_id'])
    loc_pos[loc_pos < 0] = len(index['search_locs'])
    signals['loc'] = index['loc_prior'][loc_pos][:, index['item_loc_pos']]
    center = index['search_center'].reindex(batch['search_location_id'])
    dist = haversine(center['lat'].values[:, None], center['lon'].values[:, None], index['item_lat'][None, :], index['item_lon'][None, :])
    signals['dist'] = np.nan_to_num(dist, nan=0.0).astype(np.float32)

    signals['flt'] = filter_scores(index, batch['search_infm_params_text'].values)
    signals['cat'] = np.broadcast_to(index['is_services'], signals['emb'].shape)
    return signals


def combine(s, w):
    score = w['bm_full'] * s['bm_full'] + w['bm_title'] * s['bm_title'] + w['exp'] * s['exp']
    score = score + w['emb'] * s['emb'] + w['emb_ft'] * s['emb_ft']
    local = 1 - w['loc_local'] * s['nl']
    score = score + w['loc'] * local * np.log(s['loc'] + w['loc_eps']) - w['dist'] * np.log1p(s['dist'] / w['dist_scale'])
    score = score + w['mc'] * np.log(s['mc'] + w['mc_eps']) + w['flt'] * s['flt'] + w['cat'] * s['cat']
    return score


def top_k(score, k=50):
    k = min(k, score.shape[1])
    part = np.argpartition(-score, k - 1, axis=1)[:, :k]
    order = np.argsort(-np.take_along_axis(score, part, axis=1), axis=1, kind='stable')
    return np.take_along_axis(part, order, axis=1)

Функция прогона: для батча считает сигналы один раз, а дальше для каждого набора весов берет топ-50. Если передать `pool_weights`, она еще сохраняет "пул кандидатов" для подбора весов: объединение топ-300 по нескольким разным наборам весов вместе со значениями всех сигналов. Потом веса перебираются уже только внутри пула, это в сотни раз быстрее, чем пересчитывать весь корпус.

In [30]:
SIGNALS = ['bm_full', 'bm_title', 'exp', 'emb', 'emb_ft', 'mc', 'nl', 'loc', 'dist', 'flt', 'cat']


def run(index, qdf, weight_sets, pool_weights=None, pool_size=300, batch_size=200):
    predictions = {name: [] for name in weight_sets}
    pool = []
    for start in range(0, len(qdf), batch_size):
        batch = qdf.iloc[start:start + batch_size]
        s = compute_signals(index, batch)
        for name, w in weight_sets.items():
            top = top_k(combine(s, w))
            predictions[name].extend([list(index['item_ids'][row]) for row in top])
        if pool_weights is not None:
            cand = np.concatenate([top_k(combine(s, w), pool_size) for w in pool_weights], axis=1)
            for row in range(len(batch)):
                c = np.unique(cand[row])
                part = pd.DataFrame({name: s[name][row, c] for name in SIGNALS})
                part['q'] = start + row
                part['item_id'] = index['item_ids'][c]
                pool.append(part)
    if pool_weights is not None:
        return predictions, pd.concat(pool, ignore_index=True)
    return predictions

## 9. Пошаговая проверка на валидации

Собираю индекс по обучающей части и прогоняю выборки A и B. Начальные веса выставлены вручную по порядку величин сигналов (BM25 порядка 5-20, косинусы 0-1, логарифм вероятности локации от -10 до 0). Каждая следующая строка добавляет один сигнал. Строка "первая версия" - веса, с которыми был отправлен первый ответ (0.877 на платформе).

Важно: веса в этой таблице выставлены вручную и не согласованы между собой, поэтому последние шаги (дообученные эмбеддинги и нелокальность) тут могут даже немного ухудшать результат. Реальный вклад каждого сигнала видно в таблице "без ..." после подбора весов (раздел 10).

In [31]:
t = time.time()
val_index = build_index(train_part, val_corpus, item_emb[keep], ft_item_emb)
print('индекс для валидации собран за', round(time.time() - t), 'сек')

zero = dict(bm_full=0, bm_title=0, exp=0, emb=0, emb_ft=0, loc=0, loc_eps=1e-4, loc_local=0, dist=0, dist_scale=30, mc=0, mc_eps=1e-3, flt=0, cat=0)
steps = {}
steps['1. BM25 по тексту'] = dict(zero, bm_full=1)
steps['2. + BM25 по заголовку'] = dict(steps['1. BM25 по тексту'], bm_title=0.5)
steps['3. + локация'] = dict(steps['2. + BM25 по заголовку'], loc=2)
steps['4. + фильтры'] = dict(steps['3. + локация'], flt=2)
steps['5. + подкатегория'] = dict(steps['4. + фильтры'], mc=1)
steps['6. + эмбеддинги'] = dict(steps['5. + подкатегория'], emb=10)
steps['7. + расширение запроса'] = dict(steps['6. + эмбеддинги'], exp=5)
steps['8. + расстояние'] = dict(steps['7. + расширение запроса'], dist=0.5)
steps['9. + дообученные эмбеддинги'] = dict(steps['8. + расстояние'], emb_ft=10)
steps['10. + нелокальность запроса'] = dict(steps['9. + дообученные эмбеддинги'], loc_local=1)
steps['первая версия (отправлена, 0.877)'] = dict(zero, bm_full=1, exp=5, emb=40, loc=2, loc_eps=3e-5, dist=1, dist_scale=30, mc=0.5, mc_eps=1e-3, flt=5)
pool_weights = [
    steps['3. + локация'],
    dict(zero, emb=20, loc=2),
    dict(zero, emb_ft=20, loc=2, dist=1),
    dict(zero, exp=20, loc=2, flt=2),
    steps['первая версия (отправлена, 0.877)'],
    dict(steps['первая версия (отправлена, 0.877)'], emb_ft=10, dist=2, dist_scale=10, loc_local=1),
]

t = time.time()
val_pred, pool = run(val_index, val_ab, steps, pool_weights=pool_weights)
print('прогон валидации занял', round(time.time() - t), 'сек, строк в пуле:', len(pool))

is_a = (val_ab['part'] == 'A').values
rows = []
for name, pred in val_pred.items():
    rows.append({'шаг': name,
                 'Recall@50 A': recall_at_50([p for p, f in zip(pred, is_a) if f], val_ab.loc[is_a, 'relevant']),
                 'Recall@50 B': recall_at_50([p for p, f in zip(pred, is_a) if not f], val_ab.loc[~is_a, 'relevant'])})
pd.DataFrame(rows).round(4)

индекс для валидации собран за 47 сек


прогон валидации занял 1620 сек, строк в пуле: 4902537


,шаг,Recall@50 A,Recall@50 B
0,1. BM25 по тексту,0.3380,0.3414
1,2. + BM25 по заголовку,0.3726,0.3741
2,3. + локация,0.8466,0.8302
3,4. + фильтры,0.8689,0.8501
4,5. + подкатегория,0.8841,0.8641
5,6. + эмбеддинги,0.8848,0.8635
6,7. + расширение запроса,0.8848,0.8598
7,8. + расстояние,0.8920,0.8695
8,9. + дообученные эмбеддинги,0.8872,0.8628
9,10. + нелокальность запроса,0.8819,0.8585


## 10. Подбор весов по валидации

Перебор по координатам: беру один вес, пробую несколько значений, оставляю лучшее по Recall@50 на выборке A, перехожу к следующему. Три полных прохода. Выборку B при подборе не трогаю и смотрю на нее только в конце.

Метрика внутри пула считается так же, как обычная: сортирую кандидатов запроса по скору, беру 50 лучших и делю число найденных релевантных на число всех релевантных запроса (включая те, что в пул не попали).

In [32]:
relevant_sets = [set(r) for r in val_ab['relevant']]
pool['rel'] = [item in relevant_sets[q] for q, item in zip(pool['q'], pool['item_id'])]
n_relevant = val_ab['relevant'].str.len().values
pool_q = pool['q'].values
pool_rel = pool['rel'].values
pool_signals = {name: pool[name].values for name in SIGNALS}
print('доля релевантных, которые вообще попали в пул:', round(pool['rel'].sum() / n_relevant.sum(), 4))


def pool_recall(w):
    score = combine(pool_signals, w)
    order = np.lexsort((-score, pool_q))
    q_sorted = pool_q[order]
    starts = np.r_[0, np.flatnonzero(np.diff(q_sorted)) + 1]
    rank = np.arange(len(order)) - np.repeat(starts, np.diff(np.r_[starts, len(order)]))
    hits = np.bincount(q_sorted[(rank < 50) & pool_rel[order]], minlength=len(val_ab))
    recall = hits / n_relevant
    return recall[is_a].mean(), recall[~is_a].mean()


grid = {
    'bm_full': [0.5, 1, 1.5, 2],
    'bm_title': [0, 0.25, 0.5, 1],
    'exp': [0, 2.5, 5, 10, 15],
    'emb': [0, 10, 20, 30, 40, 60],
    'emb_ft': [0, 5, 10, 20, 30, 40],
    'loc': [1, 1.5, 2, 2.5, 3],
    'loc_eps': [1e-5, 3e-5, 1e-4, 3e-4, 1e-3],
    'loc_local': [0, 0.5, 1, 1.5, 2],
    'dist': [0, 0.5, 1, 2, 3, 5],
    'dist_scale': [3, 10, 30, 100, 300],
    'mc': [0, 0.25, 0.5, 1, 2],
    'mc_eps': [1e-4, 1e-3, 1e-2],
    'flt': [0, 1, 2, 3, 5, 8],
    'cat': [0, 1, 2],
}

best_w = dict(steps['10. + нелокальность запроса'])
best_score = pool_recall(best_w)[0]
print('старт: A =', round(best_score, 4))
for iteration in range(3):
    for name, values in grid.items():
        for v in values:
            w = dict(best_w, **{name: v})
            score = pool_recall(w)[0]
            if score > best_score + 1e-6:
                best_score, best_w = score, w
    print(f'проход {iteration + 1}: A = {best_score:.4f}')
print(best_w)

доля релевантных, которые вообще попали в пул: 0.9778


старт: A = 0.8824


проход 1: A = 0.9012


проход 2: A = 0.9085


проход 3: A = 0.9112
{'bm_full': 1, 'bm_title': 0.25, 'exp': 2.5, 'emb': 10, 'emb_ft': 20, 'loc': 2.5, 'loc_eps': 3e-05, 'loc_local': 2, 'dist': 2, 'dist_scale': 10, 'mc': 0.25, 'mc_eps': 0.0001, 'flt': 5, 'cat': 2}


In [33]:
a, b = pool_recall(best_w)
rows = [{'вариант': 'все сигналы', 'A': a, 'B': b}]
for name in ['bm_full', 'bm_title', 'exp', 'emb', 'emb_ft', 'mc', 'loc', 'loc_local', 'dist', 'flt', 'cat']:
    a, b = pool_recall(dict(best_w, **{name: 0}))
    rows.append({'вариант': 'без ' + name, 'A': a, 'B': b})
pd.DataFrame(rows).round(4)

,вариант,A,B
0,все сигналы,0.9112,0.8942
1,без bm_full,0.8395,0.8338
2,без bm_title,0.9085,0.8930
3,без exp,0.9100,0.8924
4,без emb,0.9088,0.8937
5,без emb_ft,0.8889,0.8801
6,без mc,0.9089,0.8900
7,без loc,0.8002,0.7910
8,без loc_local,0.9072,0.8914
9,без dist,0.8939,0.8765


Проверка на отложенной выборке B уже полным прогоном по всему корпусу (без пула), с подобранными весами и с весами первой версии. Заодно смотрю распределение числа кандидатов, качество на виденных и новых текстах и по длине запроса.

In [34]:
t = time.time()
first_w = steps['первая версия (отправлена, 0.877)']
pred_b = run(val_index, val_b, {'best': best_w, 'first': first_w})
val_b['pred'] = pd.Series(pred_b['best'], index=val_b.index)
val_b['recall'] = [len(set(p) & set(r)) / len(r) for p, r in zip(val_b['pred'], val_b['relevant'])]
print('Recall@50 на B, первая версия:', round(recall_at_50(pred_b['first'], val_b['relevant']), 4))
print('Recall@50 на B, итоговые веса:', round(recall_at_50(pred_b['best'], val_b['relevant']), 4), '| время', round(time.time() - t), 'сек')
print('виденные тексты:', round(val_b.loc[val_b['seen'], 'recall'].mean(), 4), '| новые тексты:', round(val_b.loc[~val_b['seen'], 'recall'].mean(), 4))
print('кандидатов на запрос:', pd.Series([len(p) for p in val_b['pred']]).value_counts().to_dict())
print('запросов с дублями внутри ответа:', sum(len(p) != len(set(p)) for p in val_b['pred']))
val_b['n_words'] = val_b['search_query'].str.split().str.len().clip(upper=5)
val_b.groupby('n_words')['recall'].agg(['mean', 'size']).round(3)

Recall@50 на B, первая версия: 0.8838
Recall@50 на B, итоговые веса: 0.8942 | время 149 сек
виденные тексты: 0.9073 | новые тексты: 0.8865
кандидатов на запрос: {50: 3015}
запросов с дублями внутри ответа: 0


,mean,size
n_words,,
1,0.875,187
2,0.897,1050
3,0.905,952
4,0.874,558
5,0.903,268


Несколько примеров: запрос, релевантное объявление и первые 5 кандидатов из нашего топ-50.

In [35]:
titles = pd.Series(val_corpus['item_title_raw'].values, index=val_corpus['item_id'].values)
for _, row in val_b.sample(3, random_state=1).iterrows():
    print('ЗАПРОС:', row['search_query'], '| фильтры:', row['search_infm_params_text'] or '-', '| recall:', row['recall'])
    print('   нужно:', [titles[i] for i in row['relevant']])
    print('   топ-5:', [titles[i] for i in row['pred'][:5]])
    print()

ЗАПРОС: 3d-визуализация прихожей/входной группы | фильтры: - | recall: 1.0
   нужно: ['3D-визуализация домов, проект(рендеринг)']
   топ-5: ['3D визуализация интерьеров — быстро и с гарантией', '3D визуализация интерьера', 'Визуализатор, визуализация интерьера 3D', '3D визуализация интерьеров и корпусной мебели', '3D визуализация интерьера']

ЗАПРОС: прошивка ipad | фильтры: - | recall: 1.0
   нужно: ['Разблокировка телефонов, програмный ремонт']
   топ-5: ['Чип-тюнинг, ремонт эбу, иммобилайзер, SRS, ODO', 'Прошивка на BMW X5 G05 M50d', 'Замена Разьема Samsung A51', 'Русификация автомобилей', 'Замена задней крышки iPhone']

ЗАПРОС: мусорный контейнер подольск | фильтры: Вид услуги Вывоз мусора и вторсырья | recall: 0.0
   нужно: ['Вывоз мусора из Подольска хлам демонтаж']
   топ-5: ['Вывоз мусора самосвалом, контейнером, газелью', 'Вывоз мусора контейнером, аренда спец.техники', 'Вывоз мусора:контейнеры и газели', 'Вывоз мусора контейнер', 'Вывоз мусора']



Разбор промахов: для объявлений, которые не попали в топ-50, смотрю, насколько вероятна была их локация.

In [36]:
missed = []
for _, row in val_b.iterrows():
    for item_id in row['relevant']:
        if item_id not in set(row['pred']):
            missed.append({'query': row['search_query'], 'search_location_id': row['search_location_id'], 'item_id': item_id})
missed = pd.DataFrame(missed)
missed = missed.merge(val_corpus[['item_id', 'item_title_raw', 'item_location_id']], on='item_id', how='left')
pair_counts = train_part.groupby(['search_location_id', 'item_location_id']).size()
search_counts = train_part.groupby('search_location_id').size()
pair_n = pair_counts.reindex(list(zip(missed['search_location_id'], missed['item_location_id']))).fillna(0).values
missed['p_loc'] = pair_n / search_counts.reindex(missed['search_location_id']).fillna(1).values
print('промахов:', len(missed), 'из', val_b['relevant'].str.len().sum())
print('доля промахов, где P(локация объявления | локация поиска) < 0.01:', round((missed['p_loc'] < 0.01).mean(), 3))
print('доля промахов, где такой пары локаций вообще не было в train:', round((missed['p_loc'] == 0).mean(), 3))
missed.sample(12, random_state=0)[['query', 'item_title_raw', 'p_loc']]

промахов: 355 из 3296
доля промахов, где P(локация объявления | локация поиска) < 0.01: 0.454
доля промахов, где такой пары локаций вообще не было в train: 0.217


,query,item_title_raw,p_loc
6,обучение курсы практика,Курсы по шлифовке коленвалов,0.975904
140,аренда vesta под такси,LADA Vesta / Аренда авто под такси без залога,0.004135
301,перевозка вещей межгород,Грузоперевозки форд транспорт с грузчиками,0.924313
220,динамометрический ключ аренда,Гайковерт Аренда,0.979696
90,тюнинг салона авто мерсадес,"Установка Bi LED, ремонт фар, тюнинг фар",0.924313
229,поверка счетчиков воды,Поверка счётчиков водоснабжения,0.000000
114,металлокострукции для мойки самообслуживания,Изготовление ферм и металлоконструкций,0.982742
60,фундамент брасово,Заливка фундамента/ Заливка/ отмостки/ Сваи,0.000000
37,покраска вещей,Покраска наушников,0.944664
295,аренда кабриолета,Аренда кабриолета BMW 218,0.000067


Выводы по валидации:

- Один BM25 дает около 0.34 (на старой, легкой валидации было 0.39). Самый большой прирост по-прежнему дает локация: без нее итоговый результат падает с 0.894 до 0.79.
- На новой валидации, где рядом много похожих объявлений, стали важнее расстояние (без него 0.877) и фильтры (без них 0.882).
- Дообученные эмбеддинги - второй по важности текстовый сигнал после BM25: без них 0.880 вместо 0.894. Они лучше отличают нужное объявление от похожих в том же городе, особенно для запросов, которых не было в train.
- Нелокальность запроса дает немного (около 0.3 пункта), но в нужную сторону: для запросов про базы отдыха, перевозки между городами, онлайн-услуги штраф за "чужую" локацию ослабляется.
- Категория 114 ничего не меняет: объявления из других категорий и так проигрывают по тексту и подкатегории. Жесткого фильтра по категории нет, так что редкие категории не выкидываются совсем.
- Итог на отложенной выборке B: первая версия 0.884, новая 0.894. На текстах, которые встречались в train, 0.907, на новых 0.887.
- Остаются два типа ошибок: запросы, под которые подходят сотни объявлений в одном городе, и объявления из "необычной" локации (около 45% промахов).

Точные значения на A и B - в таблицах выше.

## 11. Финальный ответ для benchmark

Теперь индекс строится по всему train, корпус - только `benchmark_items`, веса - подобранные выше. Дообученная модель та же, что на валидации (обучена на 60% train).

In [37]:
t = time.time()
bench_corpus = all_corpus.iloc[:n_bench_items].reset_index(drop=True)
assert (val_corpus['item_id'].iloc[:n_bench_items].values == bench_corpus['item_id'].values).all()
final_index = build_index(train, bench_corpus, item_emb[:n_bench_items], ft_item_emb[:n_bench_items])
predictions = run(final_index, queries, {'best': best_w})['best']
print('предсказания готовы за', round(time.time() - t), 'сек')

answer = pd.DataFrame({
    'query_id': queries['query_id'].values,
    'answer': [' '.join(top50) for top50 in predictions],
})
print(answer.dtypes)
answer.to_csv('answer.csv', index=False)
answer.head()

предсказания готовы за 168 сек
query_id    str
answer      str
dtype: object


,query_id,answer
0,70DfDUpwjxB4lzFd,355392014208b7bf d722bcda1a555091 255fbeaf526a1cc1 6042fecdd7753112 603623b4bd9e8f6c deb58d7ad8c...
1,JTrdTaZJvSiLPkXj,422d3ffdd5bbf626 cbeccbecb1fb8d86 1c6d2079e07f4497 367af128a9ea2a48 47b2ed1012de011a 1344796085e...
2,LZCZNoVG4AFUkVRJ,3b370cc603f67947 168a9207e80b0be4 d8fce513e4f000a7 dab52187b4500d9b 3f89b8062dc85f1c edecb3695ab...
3,660ac9QVtXkRxZC3,af91ec4a29b66636 da18e5119a944f0b bd2cb4500fad728e 3cce43c811459438 4c15e7abba063ac5 a846a2a5241...
4,YgHcM9MVbxKnxD1e,1a62634394544781 9c41fe789f98d702 615c73ea4c3bfd15 60e3b99111bd032b 53f9c8caeeb8111e 13f58fe9542...


## 12. Проверки ответа (чек-лист)

In [38]:
check = pd.read_csv('answer.csv', dtype=str, keep_default_na=False, encoding='utf-8')
with open('answer.csv', encoding='utf-8') as f:
    header = f.readline().strip()
lists = check['answer'].str.split(' ')
corpus_ids = set(items['item_id'])
answer_ids = set(i for lst in lists for i in lst)

checks = {
    'заголовок ровно query_id,answer': header == 'query_id,answer',
    'ровно две колонки': check.columns.tolist() == ['query_id', 'answer'],
    'строк столько же, сколько запросов': len(check) == len(queries),
    'все query_id из benchmark есть': set(check['query_id']) == set(queries['query_id']),
    'нет лишних query_id': set(check['query_id']) <= set(queries['query_id']),
    'нет повторов query_id': check['query_id'].is_unique,
    'query_id по 16 символов': check['query_id'].str.len().eq(16).all(),
    'не больше 50 item_id в строке': lists.str.len().le(50).all(),
    'нет пустых ответов': lists.str.len().ge(1).all(),
    'нет повторов item_id в строке': all(len(x) == len(set(x)) for x in lists),
    'все item_id есть в корпусе': answer_ids <= corpus_ids,
    'item_id через один пробел, только 0-9a-f': check['answer'].str.fullmatch(r'[0-9a-f]{16}( [0-9a-f]{16})*').all(),
    'файл совпадает с тем, что было в памяти': check['query_id'].tolist() == answer['query_id'].tolist() and check['answer'].tolist() == answer['answer'].tolist(),
}
for name, ok in checks.items():
    print('OK  ' if ok else 'FAIL', name)
assert all(checks.values())
print('item_id на запрос:', lists.str.len().value_counts().to_dict())

OK   заголовок ровно query_id,answer
OK   ровно две колонки
OK   строк столько же, сколько запросов
OK   все query_id из benchmark есть
OK   нет лишних query_id
OK   нет повторов query_id
OK   query_id по 16 символов
OK   не больше 50 item_id в строке
OK   нет пустых ответов
OK   нет повторов item_id в строке
OK   все item_id есть в корпусе
OK   item_id через один пробел, только 0-9a-f
OK   файл совпадает с тем, что было в памяти
item_id на запрос: {50: 2452}


Пограничные случаи на запросах benchmark.

In [39]:
bench = queries.copy()
bench['pred'] = pd.Series(predictions, index=bench.index)
bench['n_words'] = bench['search_query'].str.split().str.len()
bench_bm25 = (query_matrix(bench['query_st'], final_index['vocab_full']) @ final_index['bm_full']).tocsr()
bench['bm25_found'] = np.diff(bench_bm25.indptr)
bench['odd_symbols'] = bench['search_query'].str.contains(r'[^a-zа-яё0-9 ]', case=False)
empty_params = bench['search_infm_params_text'] == ''

print('пустой search_infm_params_text:', empty_params.sum(), 'запросов, кандидатов у них min =', bench.loc[empty_params, 'pred'].str.len().min())
print('запросы из одного слова:', (bench['n_words'] == 1).sum(), ', кандидатов min =', bench.loc[bench['n_words'] == 1, 'pred'].str.len().min())
print('запросы, где BM25 нашел меньше 50 объявлений:', (bench['bm25_found'] < 50).sum(), ', из них совсем 0:', (bench['bm25_found'] == 0).sum())
print('   у них кандидатов min =', bench.loc[bench['bm25_found'] < 50, 'pred'].str.len().min())
print('запросы с необычными символами:', bench['odd_symbols'].sum(), ', пример:', bench.loc[bench['odd_symbols'], 'search_query'].head(5).tolist())
print('локации поиска, которых нет в train:', (~bench['search_location_id'].isin(set(train['search_location_id']))).sum())

item_category = pd.Series(items['item_category_id'].values, index=items['item_id'].values)
answer_categories = item_category[[i for p in predictions for i in p]]
print('объявлений не из категории 114 в ответах:', (answer_categories != 114).sum(), 'из', len(answer_categories))

text_key = pd.Series((items['item_title_raw'] + '|' + items['item_description_raw']).values, index=items['item_id'].values)
dup_slots = sum(len(p) - text_key[p].nunique() for p in predictions)
print('мест в топ-50, занятых копиями одинакового текста:', dup_slots, 'из', sum(len(p) for p in predictions))

пустой search_infm_params_text: 1546 запросов, кандидатов у них min = 50
запросы из одного слова: 109 , кандидатов min = 50
запросы, где BM25 нашел меньше 50 объявлений: 33 , из них совсем 0: 1
   у них кандидатов min = 50
запросы с необычными символами: 31 , пример: ['маникюр ул. выборная', 'экспертиза наро-фоминский', 'репетиторство русский,математика 4 класс', 'установка сплит-систем', 'карта т-банк 1500']
локации поиска, которых нет в train: 0


объявлений не из категории 114 в ответах: 785 из 122600


мест в топ-50, занятых копиями одинакового текста: 973 из 122600


По пограничным случаям:

- пустые фильтры и NaN в текстах не ломают пайплайн: пропуски заменены на пустые строки, у таких запросов просто нулевой сигнал `flt`;
- скор считается для всех объявлений корпуса, поэтому даже если по словам ничего не нашлось, 50 кандидатов набираются по локации, эмбеддингам и подкатегории. Повторов нет, потому что `argpartition` возвращает разные позиции корпуса, а `item_id` в корпусе уникальны;
- жесткой фильтрации по категории нет, так что редкие категории не выкидываются, они просто получают меньший скор;
- копии одинаковых объявлений занимают около 1% мест. На валидации пробовал оставлять только одну копию текста в топ-50, Recall@50 не изменился, поэтому оставил как есть;
- токенизатор на эмодзи, цифрах и латинице не падает (проверено в разделе 3).

## 13. Воспроизводимость

Случайность есть в сплите и в дообучении модели. Везде зафиксирован `SEED = 42`, порядок батчей задается через `random.Random(SEED)`, а на время обучения включен детерминированный режим PyTorch (`torch.use_deterministic_algorithms`). BM25, TF-IDF и косинусы детерминированы. Ниже хэш файла, чтобы сравнить с отправленным `answer.csv` после повторного запуска.

In [40]:
with open('answer.csv', 'rb') as f:
    print('md5 answer.csv:', hashlib.md5(f.read()).hexdigest())
print('весь ноутбук выполнился за', round((time.time() - start_time) / 60, 1), 'мин')

md5 answer.csv: 94469756af041719c372c036227ec9db
весь ноутбук выполнился за 83.8 мин


## 14. Итог

**Данные и признаки.** Из запроса: текст, локация, фильтры `search_infm_params_text`. Из объявления: заголовок, описание, параметры, подкатегория, категория, локация и координаты. Train используется для статистики (переходы "локация поиска -> локация объявления", подкатегории, заголовки и "нелокальность" по похожим запросам) и для дообучения модели эмбеддингов. Цену, рейтинг и флаги не использовал: они говорят о качестве объявления, а не о том, подходит ли оно под запрос. Проверял число отзывов как отдельный сигнал, прироста не было.

**Алгоритм.** Для каждого запроса скор считается по всему корпусу как взвешенная сумма: BM25 (весь текст и заголовок), log P(локация объявления | локация поиска) с поправкой на нелокальность запроса, штраф за расстояние, совпадение фильтров, log вероятности подкатегории по 20 похожим запросам из train, расширение запроса заголовками из train, косинус эмбеддингов готовой `rubert-tiny-turbo` и косинус эмбеддингов той же модели, дообученной на парах из train. Берутся 50 лучших.

**Как проверял.** Свой сплит train по уникальным комбинациям `search_*` полей (40% в валидацию), две выборки по ~3000 запросов с той же долей знакомых текстов, что и в benchmark (37%). Первая валидация (корпус benchmark + релевантные) оказалась слишком легкой: 0.907 на ней против 0.877 на платформе. Причина - в корпусе benchmark вокруг каждого запроса много похожих объявлений в том же городе. Новая валидация добавляет таких "похожих" объявлений для каждого запроса (раздел 6), на ней первая версия получает 0.884, что близко к платформе. Веса подбирались на A, итоговая оценка на B (разделы 9-10).

**Ошибки и пограничные случаи.** Раздел 12: формат файла и типы ID, пустые фильтры, короткие запросы, запросы без совпадений по словам, редкие категории, дубли объявлений, необычные символы. Разбор промахов в конце раздела 10.

**Что пробовал и не оставил.** TF-IDF по символьным n-граммам заголовка (ухудшал), поиск похожих запросов по символьным n-граммам вместо эмбеддингов (прироста нет), ограничение копий одинакового текста в топ-50 (прироста нет), число отзывов как сигнал (прироста нет), доля слов запроса, найденных в заголовке и тексте (прирост в пределах шума), градиентный бустинг вместо взвешенной суммы (переобучался: на каждый запрос один-два положительных примера, всего около 3000, для бустинга этого мало).

**Open-source.** pandas, numpy, scipy, scikit-learn (CountVectorizer, TfidfVectorizer), snowballstemmer, sentence-transformers + PyTorch, модель `sergeyzh/rubert-tiny-turbo` с Hugging Face.